# 03 Plan B VGGT

Goal:

1. prepare VGGT inputs from `shared/images_512/` and a subset
2. check dependencies and environment
3. run VGGT reconstruction
4. inspect sparse outputs
5. organize the standard 3DGS input structure

This notebook only covers the VGGT-based initialization pipeline.


In [22]:
from pathlib import Path
import shutil
import subprocess
import importlib

CV_ROOT = Path('/home/bzhang512/CV_Project')
DATASET_ROOT = CV_ROOT / 'dataset'
THIRD_PARTY = CV_ROOT / 'third_party'

SCENES = ['Re10k-1', 'DL3DV-2', '405841']
#SCENE = 'Re10k-1'   # 修改这里切换数据集
#SCENE = 'DL3DV-2'
SCENE = '405841'
SUBSET_NAME = 'subset_96_turnaware.txt'
VGGT_VARIANT = 'vggt_96'
OVERWRITE = False

USE_BA = True
SHARED_CAMERA = True
FINE_TRACKING = False
CAMERA_TYPE = 'SIMPLE_PINHOLE'
QUERY_FRAME_NUM = 4
MAX_QUERY_PTS = 1024

SCENE_ROOT = DATASET_ROOT / SCENE
SHARED_ROOT = SCENE_ROOT / 'part1' / 'shared'
IMAGES_512_DIR = SHARED_ROOT / 'images_512'
SUBSETS_DIR = SHARED_ROOT / 'subsets'
PLAN_B_ROOT = SCENE_ROOT / 'part1' / 'planB'
WORK_DIR = PLAN_B_ROOT / VGGT_VARIANT
VGGT_REPO = THIRD_PARTY / 'vggt'
SCENE_INPUT_DIR = WORK_DIR / 'inputs' / 'scene'
SCENE_IMAGES_DIR = SCENE_INPUT_DIR / 'images'
SCENE_SPARSE_DIR = SCENE_INPUT_DIR / 'sparse'

print('WORK_DIR =', WORK_DIR)
print('VGGT_REPO =', VGGT_REPO)
print('SCENE_INPUT_DIR =', SCENE_INPUT_DIR)


WORK_DIR = /home/bzhang512/CV_Project/dataset/405841/part1/planB/vggt_96
VGGT_REPO = /home/bzhang512/CV_Project/third_party/vggt
SCENE_INPUT_DIR = /home/bzhang512/CV_Project/dataset/405841/part1/planB/vggt_96/inputs/scene


## 1. Dependency and environment check


In [23]:
import torch
print('torch =', torch.__version__)

for name in ['hydra', 'ninja', 'pycolmap', 'trimesh']:
    try:
        importlib.import_module(name)
        print('[OK]', name)
    except Exception as e:
        print('[FAIL]', name, e)

try:
    import lightglue
    print('[OK] lightglue')
except Exception as e:
    print('[FAIL] lightglue', e)


torch = 2.7.0+cu128
[OK] hydra
[OK] ninja
[OK] pycolmap
[OK] trimesh
[OK] lightglue


## 2. Prepare the VGGT input workspace


In [24]:
for d in [WORK_DIR, WORK_DIR / 'inputs', WORK_DIR / 'raw_vggt', WORK_DIR / 'converted_colmap', WORK_DIR / 'eval', WORK_DIR / 'gs_scene']:
    d.mkdir(parents=True, exist_ok=True)

subset_file = SUBSETS_DIR / SUBSET_NAME
selected = [line.strip() for line in subset_file.read_text().splitlines() if line.strip()]
SCENE_IMAGES_DIR.mkdir(parents=True, exist_ok=True)

for name in selected:
    src = IMAGES_512_DIR / name
    dst = SCENE_IMAGES_DIR / name
    if dst.exists() and not OVERWRITE:
        continue
    shutil.copy2(src, dst)

print('prepared VGGT images:', len(selected))
print('scene images dir:', SCENE_IMAGES_DIR)


prepared VGGT images: 96
scene images dir: /home/bzhang512/CV_Project/dataset/405841/part1/planB/vggt_96/inputs/scene/images


## 3. VGGT reconstruction arguments


In [25]:
VGGT_ARGS = {
    'shared_camera': SHARED_CAMERA,
    'camera_type': CAMERA_TYPE,
    'use_ba': USE_BA,
    'fine_tracking': FINE_TRACKING,
    'query_frame_num': QUERY_FRAME_NUM,
    'max_query_pts': MAX_QUERY_PTS,
}
VGGT_ARGS


{'shared_camera': True,
 'camera_type': 'SIMPLE_PINHOLE',
 'use_ba': True,
 'fine_tracking': False,
 'query_frame_num': 4,
 'max_query_pts': 1024}

## 4. Run VGGT reconstruction


In [26]:
cmd = [
    'python', 'demo_colmap.py',
    '--scene_dir', str(SCENE_INPUT_DIR),
    '--camera_type', CAMERA_TYPE,
    '--query_frame_num', str(QUERY_FRAME_NUM),
    '--max_query_pts', str(MAX_QUERY_PTS),
]
if USE_BA:
    cmd.append('--use_ba')
if SHARED_CAMERA:
    cmd.append('--shared_camera')
if FINE_TRACKING:
    cmd.append('--fine_tracking')

print('Running command:')
print(' '.join(cmd))
subprocess.run(cmd, cwd=VGGT_REPO, check=True)
print('Expected sparse output:', SCENE_SPARSE_DIR)


Running command:
python demo_colmap.py --scene_dir /home/bzhang512/CV_Project/dataset/405841/part1/planB/vggt_96/inputs/scene --camera_type SIMPLE_PINHOLE --query_frame_num 4 --max_query_pts 1024 --use_ba --shared_camera
Arguments: {'scene_dir': '/home/bzhang512/CV_Project/dataset/405841/part1/planB/vggt_96/inputs/scene', 'seed': 42, 'use_ba': True, 'max_reproj_error': 8.0, 'shared_camera': True, 'camera_type': 'SIMPLE_PINHOLE', 'vis_thresh': 0.2, 'query_frame_num': 4, 'max_query_pts': 1024, 'fine_tracking': False, 'conf_thres_value': 5.0}
Setting seed as: 42
Using device: cuda
Using dtype: torch.bfloat16
Model loaded
[after loading model] allocated=4.69 GB, reserved=4.71 GB
Loaded 96 images from /home/bzhang512/CV_Project/dataset/405841/part1/planB/vggt_96/inputs/scene/images
[after loading images] allocated=5.81 GB, reserved=5.84 GB


/home/bzhang512/CV_Project/third_party/vggt/demo_colmap.py:81: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=dtype):


[after running VGGT aggregator] allocated=30.26 GB, reserved=38.45 GB
[after running VGGT camera head] allocated=30.26 GB, reserved=38.46 GB
[after converting pose encoding to extrinsic and intrinsic] allocated=30.26 GB, reserved=38.46 GB
[after running VGGT depth head] allocated=30.45 GB, reserved=33.96 GB
[after processing VGGT outputs] allocated=30.26 GB, reserved=33.96 GB
[after VGGT] allocated=5.82 GB, reserved=33.96 GB
[before processing reconstruction BA or w/o BA] allocated=5.82 GB, reserved=33.96 GB


/home/bzhang512/CV_Project/third_party/vggt/demo_colmap.py:161: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=dtype):
/home/bzhang512/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/home/bzhang512/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/home/bzhang512/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Predicting tracks for query frame 0
Predicting tracks for query frame 33
Predicting tracks for query frame 1
Predicting tracks for query frame 2


I20260327 03:48:45.372648 913724 misc.cc:198] 
Global bundle adjustment
I20260327 03:48:53.608610 913724 misc.cc:205] 
Bundle adjustment report
------------------------
I20260327 03:48:53.608738 913724 bundle_adjustment.cc:942] 
    Residuals : 282126
   Parameters : 17520
   Iterations : 24
         Time : 8.05841 [s]
 Initial cost : 1.79217 [px]
   Final cost : 0.432176 [px]
  Termination : Convergence

I20260327 03:48:53.608753 913724 timer.cc:91] Elapsed time: 0.137 [minutes]


[after processing reconstruction BA or w/o BA] allocated=5.82 GB, reserved=8.55 GB
Saving reconstruction to /home/bzhang512/CV_Project/dataset/405841/part1/planB/vggt_96/inputs/scene/sparse
Expected sparse output: /home/bzhang512/CV_Project/dataset/405841/part1/planB/vggt_96/inputs/scene/sparse


## 5. Check sparse outputs


In [27]:
for p in [SCENE_INPUT_DIR, SCENE_IMAGES_DIR, SCENE_SPARSE_DIR, WORK_DIR / 'raw_vggt', WORK_DIR / 'converted_colmap', WORK_DIR / 'gs_scene']:
    print("\n", p)
    if p.exists():
        for child in sorted(p.iterdir())[:20]:
            print('  ', child.name)
    else:
        print('  missing')



 /home/bzhang512/CV_Project/dataset/405841/part1/planB/vggt_96/inputs/scene
   images
   sparse

 /home/bzhang512/CV_Project/dataset/405841/part1/planB/vggt_96/inputs/scene/images
   000081.png
   000082.png
   000083.png
   000085.png
   000086.png
   000087.png
   000088.png
   000090.png
   000091.png
   000092.png
   000093.png
   000095.png
   000096.png
   000097.png
   000098.png
   000099.png
   000101.png
   000102.png
   000103.png
   000104.png

 /home/bzhang512/CV_Project/dataset/405841/part1/planB/vggt_96/inputs/scene/sparse
   cameras.bin
   images.bin
   points.ply
   points3D.bin

 /home/bzhang512/CV_Project/dataset/405841/part1/planB/vggt_96/raw_vggt

 /home/bzhang512/CV_Project/dataset/405841/part1/planB/vggt_96/converted_colmap

 /home/bzhang512/CV_Project/dataset/405841/part1/planB/vggt_96/gs_scene


## 6. Organize the standard 3DGS scene


In [28]:
gs_images_dir = WORK_DIR / 'gs_scene' / 'images'
gs_sparse0_dir = WORK_DIR / 'gs_scene' / 'sparse' / '0'
gs_images_dir.mkdir(parents=True, exist_ok=True)
gs_sparse0_dir.mkdir(parents=True, exist_ok=True)

for src in sorted(SCENE_IMAGES_DIR.iterdir()):
    if src.is_file():
        dst = gs_images_dir / src.name
        if dst.exists() and not OVERWRITE:
            continue
        shutil.copy2(src, dst)

for ext in ['*.bin', '*.txt']:
    for src in sorted(SCENE_SPARSE_DIR.glob(ext)):
        dst = gs_sparse0_dir / src.name
        if dst.exists() and not OVERWRITE:
            continue
        shutil.copy2(src, dst)

print('VGGT gs_scene ready at', WORK_DIR / 'gs_scene')
print('images:', len(list(gs_images_dir.iterdir())) if gs_images_dir.exists() else 0)
print('sparse files:', [p.name for p in sorted(gs_sparse0_dir.iterdir())])


VGGT gs_scene ready at /home/bzhang512/CV_Project/dataset/405841/part1/planB/vggt_96/gs_scene
images: 96
sparse files: ['cameras.bin', 'images.bin', 'points3D.bin']
